In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import sempler
import sempler.generators
import sempler.plot
import src.utils as utils
import src.metrics as metrics

import gnies.utils

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import gridspec

import seaborn as sns

import pickle
import os


import time

/home/juan/anaconda3/lib/python3.7/site-packages/rpy2/robjects/vectors.py:1008: UserWarning: R object inheriting from "POSIXct" but without attribute "tzone".
  warnings.warn('R object inheriting from "POSIXct" but without '
/home/juan/anaconda3/lib/python3.7/site-packages/pandas/core/arrays/datetimes.py:2192: PytzUsageWarning: The zone attribute is specific to pytz's interface; please migrate to a new time zone provider. For more details on how to do so, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  values, tz_parsed = conversion.datetime_to_datetime64(data.ravel("K"))


## Plotting setup

In [ ]:
# Set tex formatting for plots
from matplotlib import rc
rc('font',**{'family':'serif','sans-serif':['Computer Modern Roman']})
rc('text', usetex=True)
#plt.rcParams["font.family"] = "serif"
#plt.rcParams["font.serif"] = ["Computer Modern Roman"]

# Set legend size
from matplotlib.font_manager import FontProperties
fontP = FontProperties()
fontP.set_size('medium')

# Colors
def to_rgb(H, b=1, a=1):
    RGBa = []
    for h in H:
        h = h.lstrip("#")
        RGBa.append(tuple(int(h[i:i+2], 16) / 256 * b for i in (0, 2, 4)) + (a,))
    return np.array(RGBa)

cmap = matplotlib.cm.get_cmap('tab20')
# Colorblind safe palettes
base = ['#d73027', '#f46d43', '#fdae61', '#fee090', '#020202', '#abd9e9', '#74add1', '#4575b4']
#base = ['#b2182b', '#d6604d', '#f4a582', '#fddbc7', '#d1e5f0', '#92c5de', '#4393c3', '#2166ac']
#base = ['#8c510a', '#bf812d', '#dfc27d', '#f6e8c3', '#c7eae5', '#80cdc1', '#35978f', '#01665e']
# Rainbow palette
base = ["#ff4365", "#ffdd43", "#59ff43", "#43ffdd", "#7395ff", "#4365ff", "#e943ff", "#601e9e", "#6a6a6a", "#964b00"]#np.array([cmap(i) for i in range(2,20)])
plt.scatter(np.arange(len(base)), np.ones(len(base)), c = base)
#base = [base[i] for i in [0,1,3]]
colors = to_rgb(base)
colorsa = to_rgb(base, a=0.5)
colorsb = to_rgb(base, b=0.7)
plt.scatter(np.arange(len(colors)), np.zeros(len(colors)), c = colors)
plt.scatter(np.arange(len(colors)), np.ones(len(colors))*0.5, c = colorsa)
plt.scatter(np.arange(len(colors)), np.ones(len(colors))*-0.5, c = colorsb)

In [ ]:
cred = "#FF6B6B"
cgreen = "#6BCB77"
cblue = "#4D96FF"

In [ ]:
d = utils.load_bin("/home/juan/ETH/gnies-paper/light_tunnel_experiments/dataset_1718903798_light_tunnel_tag:angles/test_case_n:100_g:0_r:0")
sns.kdeplot(d[0][:,-1])
sns.kdeplot(d[1][:,-1])
sns.kdeplot(d[2][:,-1])

In [ ]:
sns.kdeplot(d[0][:,-3])
sns.kdeplot(d[1][:,-3])
sns.kdeplot(d[2][:,-3])

## To transform graph adjacencies into tikz graphs

In [ ]:
def edges_to_adjacency(edges, variables):
    """Given edges encoded as a list of tuples, i.e. (from,to), encode as
    an adjacency matrix where `A[from,to] != 0` implies `from -> to`.
    """
    p = len(variables)
    idx = dict(zip(variables, range(p)))
    A = np.zeros((p, p))
    for fro, to in edges:
        if fro in variables and to in variables:
            fro = idx[fro]
            to = idx[to]
            A[fro, to] = 1
    return A


def points_on_circle(n, radius=1):
    """
    Coordinates of n equidistant points on a circle of given radius centered at (0,0).
    """
    theta = np.linspace(0, 2 * np.pi, n + 1)
    x = np.sin(theta)[:-1]
    y = np.cos(theta)[:-1]
    points = list(zip(x, y))
    points.reverse()
    return points

def graph_to_tikz(
    A, points=None, radius=1, labels=None, spaces=4, bend=None, undirected=False, weights=False
):
    """
    Transform an adjacency matrix A, where `A[i,j] != 0` implies `i -> j` into a latex tikz graph.
    """
    p = len(A)
    if points is None:
        points = points_on_circle(p, radius)
    if labels is None:
        labels = ["$X_{%d}$" % i for i in range(p)]
    indent = " " * spaces
    string = indent + "\\begin{tikzpicture}" + "\n"
    # Add nodes
    for j, (x, y) in enumerate(points):
        string += (
            indent
            + "    \\node[circle, inner sep=0.12em] (%d) at (%0.3f, %0.3f) {%s};\n"
            % (j, x * radius, y * radius, labels[j])
        )
    # Add edges
    string += indent + "    \\begin{scope}[]" + "\n"
    for j, row in enumerate(A):
        for i in np.where(row != 0)[0]:            
            # Set tip
            if undirected and A[i, j] != 0:
                tip = "-, %s" % undirected
            else:
                tip = "->"
            # Set edge label
            if weights == 'color':
                c = 1-A[j,i]
                label = f"edge[color={{rgb,1:red,{c};green,{c};blue,{c}}}]"
            elif weights:
                label = f"{tip} node[pos=0.5, fill=white] {{\\tiny {A[j,i]:.2f}}}"
            else:
                label = "edge"
            # Set bend
            if bend is None:
                string += indent + "        \\draw[%s] (%d) %s (%d);\n" % (tip, j, label, i)
            else:
                jj, ii = 0, (i - j) % p
                direction = "left" if (ii - jj) < p / 2 else "right"
                #amount = abs(bend * (1 - (ii - jj) / p * 2))
                amount = bend * (1 - (ii - jj) / p * 2)
                bend_str = "bend %s=%d" % (direction, amount)
                string += indent + "        \\draw[%s, %s] (%d) %s (%d);\n" % (
                    tip,
                    bend_str,
                    j,
                    label,
                    i,
                )
    string += indent + "    \\end{scope}" + "\n"
    string += indent + "\\end{tikzpicture}" + "\n"
    return string

## Visualize data

In [ ]:
from causalchamber.datasets import Dataset

# Download the dataset and store it, e.g., in the current directory
dataset = Dataset('lt_interventions_standard_v1', root='./', download=True)

In [ ]:
n = 500
df_ref = dataset.get_experiment(name='uniform_reference').as_pandas_dataframe().sample(n=n, random_state=42)
df_red = dataset.get_experiment(name='uniform_red_strong').as_pandas_dataframe().sample(n=n, random_state=42)
df_blue = dataset.get_experiment(name='uniform_blue_strong').as_pandas_dataframe().sample(n=n, random_state=42)
df_green = dataset.get_experiment(name='uniform_green_strong').as_pandas_dataframe().sample(n=n, random_state=42)
df_pol_1 = dataset.get_experiment(name='uniform_pol_1_strong').as_pandas_dataframe().sample(n=n, random_state=42)
df_pol_2 = dataset.get_experiment(name='uniform_pol_2_strong').as_pandas_dataframe().sample(n=n, random_state=42)
df_angle_1 = dataset.get_experiment(name='uniform_v_angle_1_mid').as_pandas_dataframe().sample(n=n, random_state=42)
df_angle_2 = dataset.get_experiment(name='uniform_v_angle_2_mid').as_pandas_dataframe().sample(n=n, random_state=42)

In [ ]:
dataset.available_experiments()

In [ ]:
import pandas as pd
pd.unique(df_angle_1.angle_1)

In [ ]:
sns.kdeplot(df_angle_1.angle_1 - df_angle_1.angle_1.mean())
sns.kdeplot(df_ref.angle_1 - df_ref.angle_1.mean())

In [ ]:
import seaborn as sns

In [ ]:
sns.kdeplot(df_ref.red, color=cred)
sns.kdeplot(df_red.red, color=cred, linestyle='--')

sns.kdeplot(df_ref.green, color=cgreen)
sns.kdeplot(df_green.green, color=cgreen, linestyle='--')

sns.kdeplot(df_ref.blue, color=cblue)
sns.kdeplot(df_blue.blue, color=cblue, linestyle='--')

sns.kdeplot(df_ref.pol_1, color='black')
sns.kdeplot(df_pol_1.pol_1, color='black', linestyle='--')

In [ ]:
plt.figure(figsize=(1.7,1.7), dpi=200)
kwargs = {'s': 2, 'marker': '.'}

x = np.hstack([
    df_ref[['red', 'green', 'blue']].values.sum(axis=1),
    df_red[['red', 'green', 'blue']].values.sum(axis=1),
    df_green[['red', 'green', 'blue']].values.sum(axis=1),
    df_blue[['red', 'green', 'blue']].values.sum(axis=1),
    df_pol_1[['red', 'green', 'blue']].values.sum(axis=1)  
])

y = np.hstack([df_ref.ir_3,
               df_red.ir_3,
               df_green.ir_3,
               df_blue.ir_3,
               df_pol_1.ir_3])

c = np.array(['gray'] * len(df_ref) + [cred] * len(df_red) + [cgreen] * len(df_green) + [cblue] * len(df_blue) + ['black'] * len(df_pol_1))
idx = np.random.permutation(len(x))
x = x[idx]
y = y[idx]
c = c[idx]
plt.scatter(x,y,c=c, **kwargs)

plt.yticks([0,1000,2250],[])
plt.xticks([0,250,500], [])

plt.xlabel("$R + G + B$")
plt.ylabel(r"$\tilde{I}_3$", rotation=0, va="center")

# Build legend
method_entries = [Line2D([0], [0],
                         linewidth=0,
                         linestyle=None,
                         marker='.',
                         color=c) for c in ['gray', cred, cgreen, cblue, 'black']]
method_str = [r'$I = \emptyset$',
              r'$I = \{R\}$',
              r'$I = \{G\}$',
              r'$I = \{B\}$',
              r'$I = \{\theta_1\}$',
             ]
plt.savefig("figures/scatter_lt.pdf", bbox_inches="tight")

## Plot & graphs for $\mathcal{I}^\star = \emptyset$

In [ ]:
directory = "light_tunnel_experiments/dataset_1718986239_light_tunnel_tag:ref/"

test_cases = utils.read_pickle(directory + 'test_cases.pickle')
Ns = sorted(test_cases['Ns'])

**GnIES**

In [ ]:
gnies_args, gnies_results = utils.read_pickle(directory + "compiled_results_gnies_fb.pickle")
ground_truth, gnies_metrics = utils.read_pickle(directory + "metrics_gnies_fb.pickle")
gnies_lambdas = gnies_args[2]
print(gnies_lambdas)
print(gnies_metrics[metrics.success_metric].mean())

gnies_x = np.nanmean(gnies_metrics[metrics.type_1_structc], axis=(0,3))
gnies_y = np.nanmean(gnies_metrics[metrics.type_2_structc], axis=(0,3))

**GnIES w. means**

In [ ]:
gniesm_args, gniesm_results = utils.read_pickle(directory + "compiled_results_gnies_fb_means.pickle")
ground_truth, gniesm_metrics = utils.read_pickle(directory + "metrics_gnies_fb_means.pickle")
gniesm_lambdas = gniesm_args[2]
print(gniesm_lambdas)
print(gniesm_metrics[metrics.success_metric].mean())

gniesm_x = np.nanmean(gniesm_metrics[metrics.type_1_structc], axis=(0,3))
gniesm_y = np.nanmean(gniesm_metrics[metrics.type_2_structc], axis=(0,3))

**UT-IGSP**

In [ ]:
ut_igsp_args, ut_igsp_results = utils.read_pickle(directory + "compiled_results_ut_igsp_gauss_obs:0.pickle")
ground_truth, utigsp_metrics = utils.read_pickle(directory + "metrics_ut_igsp_gauss_obs:0.pickle")
utigsp_alphas, utigsp_betas = ut_igsp_args[1], ut_igsp_args[2]
print(utigsp_alphas, utigsp_betas)
print(utigsp_metrics[metrics.success_metric].mean(axis=(0,1,2,4)))
print(utigsp_metrics[metrics.type_1_structc].shape)

**UT-IGSP with HSIC tests**

In [ ]:
ut_igspH_args, ut_igspH_results = utils.read_pickle(directory + "compiled_results_ut_igsp_hsic_obs:0.pickle")
ground_truth, utigspH_metrics = utils.read_pickle(directory + "metrics_ut_igsp_hsic_obs:0.pickle")
utigspH_alphas, utigspH_betas = ut_igspH_args[1], ut_igspH_args[2]
print(utigspH_alphas, utigspH_betas)
print(utigspH_metrics[metrics.success_metric].mean(axis=(0,1,2,4)))
print(utigspH_metrics[metrics.type_1_structc].shape)

idx = list(range(len(utigspH_alphas)))
utigspH_x = np.nanmean(utigspH_metrics[metrics.type_1_structc], axis=(0,4))[idx,[0]]
utigspH_y = np.nanmean(utigspH_metrics[metrics.type_2_structc], axis=(0,4))[idx,[0]]

**GES**

In [ ]:
ges_args, ges_results = utils.read_pickle(directory + "compiled_results_ges.pickle")
ground_truth, ges_metrics = utils.read_pickle(directory + "metrics_ges.pickle")
ges_lambdas = ges_args[2]
print(ges_lambdas)
print(ges_metrics[metrics.success_metric].mean())

ges_x = np.nanmean(ges_metrics[metrics.type_1_structc], axis=(0,3))
ges_y = np.nanmean(ges_metrics[metrics.type_2_structc], axis=(0,3))

In [ ]:
# -------------------------------------------------------------------
# Plot
text = True
textsize = 5
lineopts = {'linewidth': 1}
ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
marker = {'gnies': '.',
          'gnies_means': '.',
          'gnies_rank': '.',
          'ges': '*',
          'ut_igsp': '^',
          'ut_igsp+': '^',
          'sort': 's',
          'sortp': 'X',           
}
style = {'gnies': '-',
         'gnies_means': '--',
         'gnies_rank': ':',
         'ges': ':',
         'ut_igsp': ':',
         'ut_igsp+': '--',
         'sort': ':',
         'sortp': '--',
           
}
color = {'gnies': colors[0],
         'gnies_means': colors[0],
         'gnies_rank': colorsa[0],
         'ges': colors[1],
         'ut_igsp': colors[4],
         'ut_igsp+': colors[5],
         'sort': colors[2],
         'sortp': colors[2],
         
}

print_names = {'gnies': 'GnIES',
               #'gnies_means': 'GnIES means',
               #'gnies_rank': 'GnIES-rank',
               'ges': 'GES',
               'ut_igsp': 'UT-IGSP',
               'ut_igsp+': 'UT-IGSP*',
               #'sort': 'sortnregress',         
               #'sortp': 'sortnregress',
}

def plot_metric(ax, values_x, values_y, lambdas, method, points, text, gray=False):
    c = '#aaaaaa' if gray else color[method]
    values_y = 1 - values_y
    ax.plot(values_x, values_y, color=c, linestyle=style[method], **lineopts)
    for j, l in enumerate(lambdas):
        #ax.scatter(values_x[j], values_y[j], color=color[method], marker=".", linewidth=0)
        if j==0 and text[0] is not None:
            if l < 0.001:
                l = np.log10(l)
                fmt = "$10^{%d}$"
            else:
                fmt = "$"+text[0]+"$"
            ax.text(values_x[j], values_y[j], fmt % l, fontsize=textsize, ha="left")
        if l==lambdas[-1] and text[1] is not None:
            ax.text(values_x[j], values_y[j], ("$"+text[1]+"$") % l, fontsize=textsize, ha="left")
        if j in points:
            ax.scatter(values_x[j], values_y[j], color=c, marker=marker[method], linewidth=0)
            
def set_ax(ax, yticks=True):
    ax.set_xlim([0,1])
    ax.set_ylim([0,1])
    ax.set_xlabel('FDP')    
    ax.set_ylabel('TDP') if yticks else None
    ax.set_yticks(ticks)
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticks)
    ax.set_yticklabels(ticks) if yticks else ax.set_yticklabels([])


gs = gridspec.GridSpec(1, 1, wspace=0.10, hspace=0.2)
plt.figure(figsize=(1.7,1.7), dpi=150)
ax = plt.gca()

for i,n in enumerate(Ns):
    plt.subplot(gs[i])
    ax = plt.gca()
    if i==0:
        ax0 = ax
    
    # Plot GES
    plot_metric(ax, ges_x[:,i], ges_y[:,i], ges_lambdas, 'ges', [0,2,len(ges_lambdas)-1], ["%0.2f","%d"])    
    
    # Plot GnIES
    plot_metric(ax, gnies_x[:,i], gnies_y[:,i], gnies_lambdas, 'gnies', [0,2,len(gnies_lambdas)-1], ["%0.2f","%d"])
    
    # Plot GnIES means
    #plot_metric(ax, gniesm_x[:,i], gniesm_y[:,i], gniesm_lambdas, 'gnies_means', [0,2,len(gniesm_lambdas)-1], [None if n==10 else "%0.2f","%0.1f"])
                
    # Plot UT-IGSP HSIC
    plot_metric(ax, utigspH_x[:,i], utigspH_y[:,i], utigspH_alphas, 'ut_igsp+', [0,len(utigsp_alphas)-1], [None, None])
    
    # Plot UT-IGSP
    plot_metric(ax, utigsp_x[:,i], utigsp_y[:,i], utigsp_alphas, 'ut_igsp', [0,len(utigsp_alphas)-1], ["%0.3f","%0.1f"])

    
    
    set_ax(ax, yticks=i==0)
    ax.set_title("%d obs./environment" % n)    
    ax.set_title(r"$\mathcal{I}^\star = \emptyset$", fontsize=9)


# Build legend
method_entries = [Line2D([0], [0],
                         linewidth=1,
                         linestyle=style[method],
                         marker=marker[method],
                         color=color[method]) for method in print_names.keys()]
method_str = list(print_names.values())
ax.legend(method_entries, #+ sample_size_entries
          method_str, # + sample_size_str
          prop={'size':6},
          loc='lower left',
        ncol=1)

plt.grid(color='#dddddd')
plt.savefig('figures/figure_lt_ref.pdf', bbox_inches='tight')

### Graphs

#### Ground truth

In [ ]:
import causalchamber.ground_truth as gt
from causalchamber.ground_truth import latex_name

variables=test_cases['variables']

true_dag = gt.graph('lt', 'standard').loc[variables, variables].values
print(graph_to_tikz(true_dag, radius=1.5, bend=10, labels=[latex_name(v) for v in variables], weights="color"))

#### GnIES

In [ ]:
print(gnies_lambdas, Ns)
i = 2
j = 0
print(f"lambda={gnies_lambdas[i]}, n={Ns[j]}")
estimates = gnies_results['estimates'][0,i,j,:]
dags = []
for e in estimates:
    dags += list(gnies.utils.all_dags(e))        
    average = np.sum(dags, axis=0) / len(dags)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        

print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))

#### GES

In [ ]:
print(ges_lambdas, Ns)
i = 2
j = 0
print(f"lambda={ges_lambdas[i]}, n={Ns[j]}")
estimates = ges_results['estimates'][0,i,j,:]
dags = []
for e in estimates:
    dags += list(gnies.utils.all_dags(e))        
    average = np.sum(dags, axis=0) / len(dags)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        

print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))

#### UT-IGSP

In [ ]:
ut_igsp_args, ut_igsp_results
estimates = ut_igsp_results['estimates'][0,0,0,0,:]       
average = np.sum(estimates, axis=0) / len(estimates)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        
from causalchamber.ground_truth import latex_name
print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))

## Plot & graphs for $\mathcal{I}^\star = \{\tilde{\theta}_1, \tilde{\theta}_2\}$

In [ ]:
directory = "light_tunnel_experiments/dataset_1718986313_light_tunnel_tag:angles/"

test_cases = utils.read_pickle(directory + 'test_cases.pickle')
Ns = sorted(test_cases['Ns'])

**GnIES**

In [ ]:
gnies_args, gnies_results = utils.read_pickle(directory + "compiled_results_gnies_fb.pickle")
ground_truth, gnies_metrics = utils.read_pickle(directory + "metrics_gnies_fb.pickle")
gnies_lambdas = gnies_args[2]
print(gnies_lambdas)
print(gnies_metrics[metrics.success_metric].mean())

gnies_x = np.nanmean(gnies_metrics[metrics.type_1_structc], axis=(0,3))
gnies_y = np.nanmean(gnies_metrics[metrics.type_2_structc], axis=(0,3))

**GnIES w. means**

In [ ]:
gniesm_args, gniesm_results = utils.read_pickle(directory + "compiled_results_gnies_fb_means.pickle")
ground_truth, gniesm_metrics = utils.read_pickle(directory + "metrics_gnies_fb_means.pickle")
gniesm_lambdas = gniesm_args[2]
print(gniesm_lambdas)
print(gniesm_metrics[metrics.success_metric].mean())

gniesm_x = np.nanmean(gniesm_metrics[metrics.type_1_structc], axis=(0,3))
gniesm_y = np.nanmean(gniesm_metrics[metrics.type_2_structc], axis=(0,3))

**UT-IGSP**

In [ ]:
ut_igsp_args, ut_igsp_results = utils.read_pickle(directory + "compiled_results_ut_igsp_gauss_obs:0.pickle")
ground_truth, utigsp_metrics = utils.read_pickle(directory + "metrics_ut_igsp_gauss_obs:0.pickle")
utigsp_alphas, utigsp_betas = ut_igsp_args[1], ut_igsp_args[2]
print(utigsp_alphas, utigsp_betas)
print(utigsp_metrics[metrics.success_metric].mean(axis=(0,1,2,4)))
print(utigsp_metrics[metrics.type_1_structc].shape)

**UT-IGSP with HSIC tests**

In [ ]:
ut_igspH_args, ut_igspH_results = utils.read_pickle(directory + "compiled_results_ut_igsp_hsic_obs:0.pickle")
ground_truth, utigspH_metrics = utils.read_pickle(directory + "metrics_ut_igsp_hsic_obs:0.pickle")
utigspH_alphas, utigspH_betas = ut_igspH_args[1], ut_igspH_args[2]
print(utigspH_alphas, utigspH_betas)
print(utigspH_metrics[metrics.success_metric].mean(axis=(0,1,2,4)))
print(utigspH_metrics[metrics.type_1_structc].shape)

idx = list(range(len(utigspH_alphas)))
utigspH_x = np.nanmean(utigspH_metrics[metrics.type_1_structc], axis=(0,4))[idx,[0]]
utigspH_y = np.nanmean(utigspH_metrics[metrics.type_2_structc], axis=(0,4))[idx,[0]]

**GES**

In [ ]:
ges_args, ges_results = utils.read_pickle(directory + "compiled_results_ges.pickle")
ground_truth, ges_metrics = utils.read_pickle(directory + "metrics_ges.pickle")
ges_lambdas = ges_args[2]
print(ges_lambdas)
print(ges_metrics[metrics.success_metric].mean())

ges_x = np.nanmean(ges_metrics[metrics.type_1_structc], axis=(0,3))
ges_y = np.nanmean(ges_metrics[metrics.type_2_structc], axis=(0,3))

In [ ]:
# -------------------------------------------------------------------
# Plot
text = True
textsize = 5
lineopts = {'linewidth': 1}
ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
marker = {'gnies': '.',
          'gnies_means': '.',
          'gnies_rank': '.',
          'ges': '*',
          'ut_igsp': '^',
          'ut_igsp+': '^',
          'sort': 's',
          'sortp': 'X',           
}
style = {'gnies': '-',
         'gnies_means': '--',
         'gnies_rank': ':',
         'ges': ':',
         'ut_igsp': ':',
         'ut_igsp+': '--',
         'sort': ':',
         'sortp': '--',
           
}
color = {'gnies': colors[0],
         'gnies_means': colors[0],
         'gnies_rank': colorsa[0],
         'ges': colors[1],
         'ut_igsp': colors[4],
         'ut_igsp+': colors[5],
         'sort': colors[2],
         'sortp': colors[2],
         
}

print_names = {'gnies': 'GnIES',
               #'gnies_means': 'GnIES means',
               #'gnies_rank': 'GnIES-rank',
               'ges': 'GES',
               'ut_igsp': 'UT-IGSP',
               'ut_igsp+': 'UT-IGSP*',
               #'sort': 'sortnregress',         
               #'sortp': 'sortnregress',
}

def plot_metric(ax, values_x, values_y, lambdas, method, points, text, gray=False):
    c = '#aaaaaa' if gray else color[method]
    values_y = 1 - values_y
    ax.plot(values_x, values_y, color=c, linestyle=style[method], **lineopts)
    for j, l in enumerate(lambdas):
        #ax.scatter(values_x[j], values_y[j], color=color[method], marker=".", linewidth=0)
        if j==0 and text[0] is not None:
            if l < 0.001:
                l = np.log10(l)
                fmt = "$10^{%d}$"
            else:
                fmt = "$"+text[0]+"$"
            ax.text(values_x[j], values_y[j], fmt % l, fontsize=textsize, ha="left")
        if l==lambdas[-1] and text[1] is not None:
            ax.text(values_x[j], values_y[j], ("$"+text[1]+"$") % l, fontsize=textsize, ha="left")
        if j in points:
            ax.scatter(values_x[j], values_y[j], color=c, marker=marker[method], linewidth=0)
            
def set_ax(ax, yticks=True):
    ax.set_xlim([0,1])
    ax.set_ylim([0,1])
    ax.set_xlabel('FDP')    
    ax.set_ylabel('TDP') if yticks else None
    ax.set_yticks(ticks)
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticks)
    ax.set_yticklabels(ticks) if yticks else ax.set_yticklabels([])


gs = gridspec.GridSpec(1, 1, wspace=0.10, hspace=0.2)
plt.figure(figsize=(1.7,1.7), dpi=150)
ax = plt.gca()

for i,n in enumerate(Ns):
    plt.subplot(gs[i])
    ax = plt.gca()
    if i==0:
        ax0 = ax
    
    # Plot GES
    plot_metric(ax, ges_x[:,i], ges_y[:,i], ges_lambdas, 'ges', [0,2,len(ges_lambdas)-1], ["%0.2f","%d"])    
    
    # Plot GnIES
    plot_metric(ax, gnies_x[:,i], gnies_y[:,i], gnies_lambdas, 'gnies', [0,2,len(gnies_lambdas)-1], ["%0.2f","%d"])
    
    # Plot GnIES means
    #plot_metric(ax, gniesm_x[:,i], gniesm_y[:,i], gniesm_lambdas, 'gnies_means', [0,2,len(gniesm_lambdas)-1], [None if n==10 else "%0.2f","%0.1f"])
                
    # Plot UT-IGSP HSIC
    plot_metric(ax, utigspH_x[:,i], utigspH_y[:,i], utigspH_alphas, 'ut_igsp+', [0,len(utigsp_alphas)-1], [None, None])
    
    # Plot UT-IGSP
    plot_metric(ax, utigsp_x[:,i], utigsp_y[:,i], utigsp_alphas, 'ut_igsp', [0,len(utigsp_alphas)-1], ["%0.3f","%0.1f"])

    
    
    set_ax(ax, yticks=i==0)
    ax.set_title("%d obs./environment" % n)    
    ax.set_title(r"$\mathcal{I}^\star = \{\tilde{\theta}_1, \tilde{\theta}_2\}$", fontsize=9)


# Build legend
method_entries = [Line2D([0], [0],
                         linewidth=1,
                         linestyle=style[method],
                         marker=marker[method],
                         color=color[method]) for method in print_names.keys()]
method_str = list(print_names.values())
ax.legend(method_entries, #+ sample_size_entries
          method_str, # + sample_size_str
          prop={'size':6},
          loc='lower left',
        ncol=1)

plt.grid(color='#dddddd')
plt.savefig('figures/figure_lt_angles.pdf', bbox_inches='tight')

### Graphs

#### GnIES

In [ ]:
print(gnies_lambdas, Ns)
i = 2
j = 0
print(f"lambda={gnies_lambdas[i]}, n={Ns[j]}")
estimates = gnies_results['estimates'][0,i,j,:]
dags = []
for e in estimates:
    dags += list(gnies.utils.all_dags(e))        
    average = np.sum(dags, axis=0) / len(dags)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        

print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))

#### GES

In [ ]:
print(ges_lambdas, Ns)
i = 2
j = 0
print(f"lambda={ges_lambdas[i]}, n={Ns[j]}")
estimates = ges_results['estimates'][0,i,j,:]
dags = []
for e in estimates:
    dags += list(gnies.utils.all_dags(e))        
    average = np.sum(dags, axis=0) / len(dags)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        

print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))

#### UT-IGSP

In [ ]:
ut_igsp_args, ut_igsp_results
estimates = ut_igsp_results['estimates'][0,0,0,0,:]       
average = np.sum(estimates, axis=0) / len(estimates)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        
from causalchamber.ground_truth import latex_name
print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))

## Plot & graphs for $\mathcal{I}^\star = \{\theta_1, \theta_2\}$

In [ ]:
directory = "light_tunnel_experiments/dataset_1718986276_light_tunnel_tag:pols/"

test_cases = utils.read_pickle(directory + 'test_cases.pickle')
Ns = sorted(test_cases['Ns'])

**GnIES**

In [ ]:
gnies_args, gnies_results = utils.read_pickle(directory + "compiled_results_gnies_fb.pickle")
ground_truth, gnies_metrics = utils.read_pickle(directory + "metrics_gnies_fb.pickle")
gnies_lambdas = gnies_args[2]
print(gnies_lambdas)
print(gnies_metrics[metrics.success_metric].mean())

gnies_x = np.nanmean(gnies_metrics[metrics.type_1_structc], axis=(0,3))
gnies_y = np.nanmean(gnies_metrics[metrics.type_2_structc], axis=(0,3))

**GnIES w. means**

In [ ]:
gniesm_args, gniesm_results = utils.read_pickle(directory + "compiled_results_gnies_fb_means.pickle")
ground_truth, gniesm_metrics = utils.read_pickle(directory + "metrics_gnies_fb_means.pickle")
gniesm_lambdas = gniesm_args[2]
print(gniesm_lambdas)
print(gniesm_metrics[metrics.success_metric].mean())

gniesm_x = np.nanmean(gniesm_metrics[metrics.type_1_structc], axis=(0,3))
gniesm_y = np.nanmean(gniesm_metrics[metrics.type_2_structc], axis=(0,3))

**UT-IGSP**

In [ ]:
ut_igsp_args, ut_igsp_results = utils.read_pickle(directory + "compiled_results_ut_igsp_gauss_obs:0.pickle")
ground_truth, utigsp_metrics = utils.read_pickle(directory + "metrics_ut_igsp_gauss_obs:0.pickle")
utigsp_alphas, utigsp_betas = ut_igsp_args[1], ut_igsp_args[2]
print(utigsp_alphas, utigsp_betas)
print(utigsp_metrics[metrics.success_metric].mean(axis=(0,1,2,4)))
print(utigsp_metrics[metrics.type_1_structc].shape)

**UT-IGSP with HSIC tests**

In [ ]:
ut_igspH_args, ut_igspH_results = utils.read_pickle(directory + "compiled_results_ut_igsp_hsic_obs:0.pickle")
ground_truth, utigspH_metrics = utils.read_pickle(directory + "metrics_ut_igsp_hsic_obs:0.pickle")
utigspH_alphas, utigspH_betas = ut_igspH_args[1], ut_igspH_args[2]
print(utigspH_alphas, utigspH_betas)
print(utigspH_metrics[metrics.success_metric].mean(axis=(0,1,2,4)))
print(utigspH_metrics[metrics.type_1_structc].shape)

idx = list(range(len(utigspH_alphas)))
utigspH_x = np.nanmean(utigspH_metrics[metrics.type_1_structc], axis=(0,4))[idx,[0]]
utigspH_y = np.nanmean(utigspH_metrics[metrics.type_2_structc], axis=(0,4))[idx,[0]]

**GES**

In [ ]:
ges_args, ges_results = utils.read_pickle(directory + "compiled_results_ges.pickle")
ground_truth, ges_metrics = utils.read_pickle(directory + "metrics_ges.pickle")
ges_lambdas = ges_args[2]
print(ges_lambdas)
print(ges_metrics[metrics.success_metric].mean())

ges_x = np.nanmean(ges_metrics[metrics.type_1_structc], axis=(0,3))
ges_y = np.nanmean(ges_metrics[metrics.type_2_structc], axis=(0,3))

In [ ]:
# -------------------------------------------------------------------
# Plot
text = True
textsize = 5
lineopts = {'linewidth': 1}
ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
marker = {'gnies': '.',
          'gnies_means': '.',
          'gnies_rank': '.',
          'ges': '*',
          'ut_igsp': '^',
          'ut_igsp+': '^',
          'sort': 's',
          'sortp': 'X',           
}
style = {'gnies': '-',
         'gnies_means': '--',
         'gnies_rank': ':',
         'ges': ':',
         'ut_igsp': ':',
         'ut_igsp+': '--',
         'sort': ':',
         'sortp': '--',
           
}
color = {'gnies': colors[0],
         'gnies_means': colors[0],
         'gnies_rank': colorsa[0],
         'ges': colors[1],
         'ut_igsp': colors[4],
         'ut_igsp+': colors[5],
         'sort': colors[2],
         'sortp': colors[2],
         
}

print_names = {'gnies': 'GnIES',
               #'gnies_means': 'GnIES means',
               #'gnies_rank': 'GnIES-rank',
               'ges': 'GES',
               'ut_igsp': 'UT-IGSP',
               'ut_igsp+': 'UT-IGSP*',
               #'sort': 'sortnregress',         
               #'sortp': 'sortnregress',
}

def plot_metric(ax, values_x, values_y, lambdas, method, points, text, gray=False):
    c = '#aaaaaa' if gray else color[method]
    values_y = 1 - values_y
    ax.plot(values_x, values_y, color=c, linestyle=style[method], **lineopts)
    for j, l in enumerate(lambdas):
        #ax.scatter(values_x[j], values_y[j], color=color[method], marker=".", linewidth=0)
        if j==0 and text[0] is not None:
            if l < 0.001:
                l = np.log10(l)
                fmt = "$10^{%d}$"
            else:
                fmt = "$"+text[0]+"$"
            ax.text(values_x[j], values_y[j], fmt % l, fontsize=textsize, ha="left")
        if l==lambdas[-1] and text[1] is not None:
            ax.text(values_x[j], values_y[j], ("$"+text[1]+"$") % l, fontsize=textsize, ha="left")
        if j in points:
            ax.scatter(values_x[j], values_y[j], color=c, marker=marker[method], linewidth=0)
            
def set_ax(ax, yticks=True):
    ax.set_xlim([0,1])
    ax.set_ylim([0,1])
    ax.set_xlabel('FDP')    
    ax.set_ylabel('TDP') if yticks else None
    ax.set_yticks(ticks)
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticks)
    ax.set_yticklabels(ticks) if yticks else ax.set_yticklabels([])


gs = gridspec.GridSpec(1, 1, wspace=0.10, hspace=0.2)
plt.figure(figsize=(1.7,1.7), dpi=150)
ax = plt.gca()

for i,n in enumerate(Ns):
    plt.subplot(gs[i])
    ax = plt.gca()
    if i==0:
        ax0 = ax
    
    # Plot GES
    plot_metric(ax, ges_x[:,i], ges_y[:,i], ges_lambdas, 'ges', [0,2,len(ges_lambdas)-1], ["%0.2f","%d"])    
    
    # Plot GnIES
    plot_metric(ax, gnies_x[:,i], gnies_y[:,i], gnies_lambdas, 'gnies', [0,2,len(gnies_lambdas)-1], ["%0.2f","%d"])
    
    # Plot GnIES means
    #plot_metric(ax, gniesm_x[:,i], gniesm_y[:,i], gniesm_lambdas, 'gnies_means', [0,2,len(gniesm_lambdas)-1], [None if n==10 else "%0.2f","%0.1f"])
                
    # Plot UT-IGSP HSIC
    plot_metric(ax, utigspH_x[:,i], utigspH_y[:,i], utigspH_alphas, 'ut_igsp+', [0,len(utigsp_alphas)-1], [None, None])
    
    # Plot UT-IGSP
    plot_metric(ax, utigsp_x[:,i], utigsp_y[:,i], utigsp_alphas, 'ut_igsp', [0,len(utigsp_alphas)-1], ["%0.3f","%0.1f"])

    
    
    set_ax(ax, yticks=i==0)
    ax.set_title("%d obs./environment" % n)    
    ax.set_title(r"$\mathcal{I}^\star = \{\theta_1, \theta_2\}$", fontsize=9)


# Build legend
method_entries = [Line2D([0], [0],
                         linewidth=1,
                         linestyle=style[method],
                         marker=marker[method],
                         color=color[method]) for method in print_names.keys()]
method_str = list(print_names.values())
ax.legend(method_entries, #+ sample_size_entries
          method_str, # + sample_size_str
          prop={'size':6},
          loc='lower left',
        ncol=1)

plt.grid(color='#dddddd')
plt.savefig('figures/figure_lt_pols.pdf', bbox_inches='tight')

### Graphs

#### GnIES

In [ ]:
print(gnies_lambdas, Ns)
i = 2
j = 0
print(f"lambda={gnies_lambdas[i]}, n={Ns[j]}")
estimates = gnies_results['estimates'][0,i,j,:]
dags = []
for e in estimates:
    dags += list(gnies.utils.all_dags(e))        
    average = np.sum(dags, axis=0) / len(dags)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        

print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))

#### GES

In [ ]:
print(ges_lambdas, Ns)
i = 2
j = 0
print(f"lambda={ges_lambdas[i]}, n={Ns[j]}")
estimates = ges_results['estimates'][0,i,j,:]
dags = []
for e in estimates:
    dags += list(gnies.utils.all_dags(e))        
    average = np.sum(dags, axis=0) / len(dags)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        

print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))

#### UT-IGSP

In [ ]:
ut_igsp_args, ut_igsp_results
estimates = ut_igsp_results['estimates'][0,0,0,0,:]       
average = np.sum(estimates, axis=0) / len(estimates)
sempler.plot.plot_graph(average, labels=test_cases['variables'], weights=True)
        
from causalchamber.ground_truth import latex_name
print(graph_to_tikz(average, radius=1.5, bend=10, labels=[latex_name(v) for v in test_cases['variables']], weights="color"))